# Inference and evaluation

This notebook accepts checkpoints from vision pretraining (`train_vision.py`), joint
training from scratch (`train_all_from_start.py`, formerly `train_pose.py`), and
staged fine-tuning (`train_tot.py`).

Vision checkpoints support vision plots and metrics. They do not contain a trained
pose head, so pose evaluation is skipped. Joint checkpoints support both vision and
pose evaluation. Pose inference uses predicted occupancy bins and predicted
descriptor matches; GT camera poses are used only to score the predictions. The
optional vision subset is selected using GT cylinder matches and a valid camera
baseline.


In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset

from dataset import SceneTwoPairsDataset
from model import PairImageCylinderModel, gt_bin_matches
from utils import plot_estimated_cylinders_on_images, plot_relative_pose, pose_to_text

# Point this at best_vision.pt for train_vision, or best.pt for a joint checkpoint.
CHECKPOINT = Path("runs/REPLACE_WITH_RUN/checkpoints/best_vision.pt")
TEST_DATASET = Path("testdataset")

VISION_ON_GT_POSE_SUBSET = False  # Limit vision metrics to pairs with GT pose matches.
MODEL_VERSION = "best"  # Descriptive label for plots.
INFERENCE_BATCH_SIZE = 4
VISION_SAMPLE_IDX = 0
INFERENCE_SHOW_TEXT = False
INFERENCE_SHOW_IMAGES = True
OCC_THRESHOLD = 0.4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint:", CHECKPOINT)
print("Test dataset:", TEST_DATASET)


In [ ]:
import torch.nn.functional as F

checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
training_args = checkpoint.get("args", {})
state = checkpoint["model_state_dict"]
training_stage = checkpoint.get("training_stage", "")
IS_VISION_CHECKPOINT = training_stage.startswith("vision_")
HAS_TRAINED_POSE = not IS_VISION_CHECKPOINT and "pose_head.token_mlp.0.weight" in state

model_kwargs = dict(
    img_size=training_args.get("img_size", 128),
    patch_size=tuple(training_args.get("patch_size", (16, 4))),
    in_chans=3,
    embed_dim=training_args.get("embed_dim", 192),
    depth=training_args.get("depth", 4),
    num_heads=training_args.get("num_heads", 4),
    num_bins=training_args.get("num_bins", 128),
    dropout=training_args.get("dropout", 0.1),
)

if HAS_TRAINED_POSE:
    scale = state["pose_head.input_scale"]
    model_kwargs.update(
        pose_embed_dim=state["pose_head.token_mlp.0.weight"].shape[0],
        pose_depth=len({key.split(".")[3] for key in state
                        if key.startswith("pose_head.blocks.layers.")}),
        fov_degrees=float(scale[0]) * 360 / np.pi,
        radius_scale=float(scale[1]),
        distance_scale=float(scale[2]),
    )

model = PairImageCylinderModel(**model_kwargs)
if IS_VISION_CHECKPOINT:
    # Vision pretraining saves a full model state, but its pose head is untrained.
    vision_state = {key: value for key, value in state.items()
                    if key.startswith(("backbone.", "vision_head.", "corr_decoder."))}
    expected = {key for key in model.state_dict()
                if key.startswith(("backbone.", "vision_head.", "corr_decoder."))}
    if set(vision_state) != expected:
        raise ValueError("Vision checkpoint does not match the model's vision modules.")
    model.load_state_dict(vision_state, strict=False)
else:
    model.load_state_dict(state, strict=True)
model = model.to(DEVICE).eval()
history = checkpoint.get("history")
GT_OCC_THRESHOLD = training_args.get("occ_thresh", 0.5)
print(f"Loaded checkpoint: {CHECKPOINT}")
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print("Training stage:", training_stage or "unspecified")
print("Trained pose head:", HAS_TRAINED_POSE)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
if HAS_TRAINED_POSE:
    print("Pose matching: predicted occupancy bins and descriptor matches (no GT cylinders).")
else:
    print("Vision-only checkpoint: pose evaluation will be skipped.")


In [ ]:
infer_dataset = SceneTwoPairsDataset(
    root_dir=str(TEST_DATASET),
    image_size=training_args.get("img_size", IMG_SIZE),
    return_two_pairs=False,
)
infer_loader = DataLoader(
    infer_dataset, batch_size=INFERENCE_BATCH_SIZE,
    shuffle=False, num_workers=0,
)

# Determine the same eligible subset using only GT labels, before inference.
pose_subset_indices = []
for sample_index, sample in enumerate(infer_dataset.samples):
    pair = sample.pair
    gt_a = torch.as_tensor(pair["vision1"], dtype=torch.float32)
    gt_b = torch.as_tensor(pair["vision2"], dtype=torch.float32)
    matches = gt_bin_matches(gt_a.unsqueeze(0), gt_b.unsqueeze(0), GT_OCC_THRESHOLD)[0]
    if len(matches) < 2:
        continue
    camera_a = pair.get("camera1", sample.scene_camera1)
    camera_b = pair.get("camera2", sample.scene_camera2)
    pose = infer_dataset._relative_pose_2d(camera_a, camera_b)
    if not torch.isfinite(pose).all():
        raise ValueError(f"Nonfinite GT pose in sample {sample_index}")
    if pose[:2].norm() > 1e-6:
        pose_subset_indices.append(sample_index)

vision_loader = (
    DataLoader(Subset(infer_dataset, pose_subset_indices),
               batch_size=INFERENCE_BATCH_SIZE, shuffle=False, num_workers=0)
    if VISION_ON_GT_POSE_SUBSET else infer_loader
)
print("Inference dataset size:", len(infer_dataset))
print("GT-eligible pose pairs:", len(pose_subset_indices))
print("Vision evaluated on:", "GT pose subset" if VISION_ON_GT_POSE_SUBSET else "all pairs")


In [ ]:
batch = next(iter(infer_loader))
img_a, vision_a, img_b, vision_b, pose_ab = (x.to(DEVICE) for x in batch)
model.eval()
with torch.no_grad():
    if HAS_TRAINED_POSE:
        pred_vision, pred_vision_b, pred_pose, pose_info = model(
            img_a, img_b,
            target_vision_a=vision_a,
            target_vision_b=vision_b,
            matching_mode="predicted",
            return_pose_inputs=True,
            pred_occ_thresh=OCC_THRESHOLD,
        )
    else:
        pred_vision, pred_vision_b, _ = model(
            img_a, img_b, compute_pose=False, return_corr=False,
        )
        pred_pose, pose_info = None, None
if not 0 <= VISION_SAMPLE_IDX < len(img_a):
    raise IndexError("VISION_SAMPLE_IDX is outside this batch")
pose_sample_valid = False
if HAS_TRAINED_POSE:
    pose_sample_valid = (
        pose_info["pose_valid"][VISION_SAMPLE_IDX]
        & (pose_ab[VISION_SAMPLE_IDX, :2].norm() > 1e-6)
    ).item()
print("pred_vision shape:", pred_vision.shape)
if HAS_TRAINED_POSE:
    print("pred_pose shape:", pred_pose.shape)
    print("Pose eligible for selected sample:", pose_sample_valid)


In [ ]:
from metrics import evaluate_vision

class VisionOnlyModel(torch.nn.Module):
    """Let the existing vision evaluator call forward without GT matches."""
    def __init__(self, joint_model):
        super().__init__()
        self.joint_model = joint_model

    def forward(self, img_a, img_b, **kwargs):
        kwargs["compute_pose"] = False
        return self.joint_model(img_a, img_b, **kwargs)

vision_metrics = evaluate_vision(
    VisionOnlyModel(model),
    vision_loader,
    DEVICE,
    occ_threshold=OCC_THRESHOLD,
)


In [ ]:
from metrics import evaluate_pose

if HAS_TRAINED_POSE:
    pose_metrics = evaluate_pose(
        model,
        infer_loader,
        DEVICE,
        pred_occ_threshold=OCC_THRESHOLD,
    )
else:
    pose_metrics = None
    print("Skipping pose evaluation: this checkpoint has no trained pose head.")


## 1. Input image pair


In [ ]:
# Plots are controlled by INFERENCE_SHOW_IMAGES in the following cells.


## 2. Vision prediction

Compare the predicted `(occupancy, radius, depth)` against the ground truth for image A.


In [ ]:
sample_idx = VISION_SAMPLE_IDX
bins = np.arange(vision_a.shape[1])
PLOT_OCC_THRESHOLD = OCC_THRESHOLD
if INFERENCE_SHOW_IMAGES:
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    names = ["occupancy", "radius", "depth"]
    pr_occ = pred_vision[sample_idx, :, 0].detach().cpu().numpy()
    for j, ax in enumerate(axes):
        gt = vision_a[sample_idx, :, j].detach().cpu().numpy()
        pr = pred_vision[sample_idx, :, j].detach().cpu().numpy()
        ax.plot(bins, gt, label="GT", color="tab:blue")
        if j == 0:
            ax.plot(bins, pr, label="Pred", color="tab:orange")
        else:
            pr_mask = pr_occ > PLOT_OCC_THRESHOLD
            ax.scatter(bins[pr_mask], pr[pr_mask], label="Pred", color="tab:orange", s=25)
        ax.set_ylabel(names[j])
        ax.grid(True, alpha=0.3)
        ax.legend()
    axes[-1].set_xlabel("Bin")
    fig.suptitle(f"Vision prediction sample {sample_idx}")
    plt.tight_layout()
    plt.show()


## 3. Estimated cylinders and input images


In [ ]:
if INFERENCE_SHOW_IMAGES:
    fig_cyl, axes_cyl = plot_estimated_cylinders_on_images(
        [img_a[VISION_SAMPLE_IDX], img_b[VISION_SAMPLE_IDX]],
        [pred_vision[VISION_SAMPLE_IDX], pred_vision_b[VISION_SAMPLE_IDX]],
        titles=["Image A: pred cylinders", "Image B: pred cylinders"],
        occ_threshold=PLOT_OCC_THRESHOLD,
        flip_x=True,
    )
    plt.show()
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img_a[VISION_SAMPLE_IDX].cpu().permute(1, 2, 0))
    axes[0].set_title("Image A")
    axes[0].axis("off")
    axes[1].imshow(img_b[VISION_SAMPLE_IDX].cpu().permute(1, 2, 0))
    axes[1].set_title("Image B")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


## 4. Relative pose


In [ ]:
if not HAS_TRAINED_POSE:
    print("No trained pose head in this checkpoint; pose visualization is unavailable.")
elif pose_sample_valid:
    gt_pose = pose_ab[VISION_SAMPLE_IDX].detach().cpu()
    pred_pose_sample = pred_pose[VISION_SAMPLE_IDX].detach().cpu()
    if INFERENCE_SHOW_TEXT:
        print("GT :", pose_to_text(gt_pose))
        print("Pred:", pose_to_text(pred_pose_sample))
    if INFERENCE_SHOW_IMAGES:
        plot_relative_pose(gt_pose, pred_pose_sample)
else:
    print("Selected sample has no evaluable pose; select another VISION_SAMPLE_IDX.")


## 5. Training history (optional)

Plot epoch-aligned loss curves stored in the selected checkpoint.


In [ ]:
# Plot epoch-aligned loss curves when the selected checkpoint contains history.
if isinstance(history, dict):
    epochs = history.get("epoch", [])
    val_epochs = history.get("val_epoch", [])
    train_keys = ["loss", "vision_loss", "pose_loss", "correspondence_loss",
                  "vision_occupancy", "vision_radius", "vision_depth"]
    val_keys = ["val_total", "val_vision", "val_pose", "val_correspondence",
                "val_depth_structure", "val_depth_delta"]
    plt.figure(figsize=(10, 5))
    plotted = False
    for keys, xs in [(train_keys, epochs), (val_keys, val_epochs)]:
        for key in keys:
            ys = history.get(key, [])
            if len(xs) and len(ys) == len(xs):
                plt.plot(xs, ys, label=key)
                plotted = True
    if plotted:
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training / validation history")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()
    else:
        plt.close()
        print("History present, but no matching epoch-aligned loss curves.")
else:
    print("No training history stored in the selected checkpoint.")
